# 04 — GNN Training

Train DTNetGNN (GAT) and IsolatedBaseline (MLP) on M5 data:
- 30,490 Item-Store series
- WRMSSE as primary metric
- BFloat16 on A100

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 4.1 Prepare Data

In [ ]:
from src.data.loader import load_m5
from src.gnn.dataset import build_dataloaders

dfs = load_m5()

train_loader, val_loader, test_loader = build_dataloaders(
    sales_df=dfs['sales'],
    calendar_df=dfs['calendar'],
    prices_df=dfs['prices'],
    G=None,
    batch_size=8, stride=7
)

## 4.2 Train Both Models

In [ ]:
from src.gnn.train import run_training

gnn_hist, base_hist = run_training(train_loader, val_loader, device)

## 4.3 Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gnn_hist['train_loss'], label='GNN train')
axes[0].plot(gnn_hist['val_loss'], label='GNN val')
axes[0].plot(base_hist['train_loss'], label='Baseline train', linestyle='--')
axes[0].plot(base_hist['val_loss'], label='Baseline val', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(gnn_hist['val_wrmsse'], label='GNN WRMSSE')
axes[1].plot(base_hist['val_wrmsse'], label='Baseline WRMSSE', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('WRMSSE')
axes[1].set_title('Validation WRMSSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()